In [1]:
import pandas as pd

from Bio import SeqIO
from modules.sequences import split_train_val_test
import torch

from modules.model import (
    init_transition_mat,
    init_emission_mat
)

from modules.qmodel import (
    build_hqmm,
    convert_to_torch
)

import numpy as np
from pathlib import Path

from modules.qtrain import train_hqmm
from modules.qevaluation import description_accuracy_hqmm

import modules.qplots as qplots

import pandas as pd
import matplotlib.pyplot as plt

from modules.qplots import (
    plot_da_learning_curve,
    plot_da_bars,
)

from pathlib import Path

# Check for NVIDIA GPU (CUDA)
print("CUDA Available:", torch.cuda.is_available())

# Check for Apple Silicon Mac GPU (MPS)
print("MPS Available:", torch.backends.mps.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

CUDA Available: True
MPS Available: False
Using device: cuda


In [2]:
# Read protein families
family_df = pd.read_csv("protein_families.csv")

i = 0
name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]
match_len = int(match_len)

###### SETUP
settings = {
    "tau=0.1, n=n": (0.1, match_len),
    "tau=0.1, n=n-5": (0.1, match_len - 5),
    "tau=0.1, n=n-10": (0.1, match_len - 10),
}

max_iter = 15
tol = 1e-3
batch_size = 16
alpha = 0.8

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Directories for saved models and results
model_dir = Path("saved_models/phqmm")
results_dir = Path("results/phqmm/n")

model_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
######

# Read FASTA files
recs = [
    (r.id, str(r.seq))
    for r in SeqIO.parse(
        f"sequences/{accession}.fasta", "fasta"
    )
]

# Same train/validation/test split for all configurations
train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

results = {}
histories = {}
trained_models = {}

for label, (tau, model_match_len) in settings.items():

    model_match_len = int(model_match_len)

    if model_match_len < 2:
        raise ValueError(
            f"Invalid match length: {model_match_len}"
        )

    print(
        f"\n--- {label}: "
        f"tau={tau}, match_len={model_match_len}",
        flush=True
    )

    # Initialize matrices for this match length
    trans_matdf = init_transition_mat(model_match_len)
    emit_matdf = init_emission_mat(
        model_match_len, seed=2
    )

    # Encode symbols
    symbols = list(emit_matdf.index)
    sym_idx = {
        s: j for j, s in enumerate(symbols)
    }

    # Construct HQMM for this match length
    K = build_hqmm(trans_matdf, emit_matdf)
    K_torch = convert_to_torch(K)

    K_torch = {
        key: value.detach().to(device)
        for key, value in K_torch.items()
    }

    # Actual number of HQMM hidden states
    W = next(iter(K_torch.values())).shape[0]

    # Initial density matrix
    p = torch.zeros(
        (W, W),
        dtype=torch.complex64,
        device=device
    )
    p[0, 0] = 1.0

    # Untrained test DA
    with torch.no_grad():
        da0 = description_accuracy_hqmm(
            p, K_torch, W, sym_idx, test_seqs
        )

    untrained_mean = float(da0.mean())
    untrained_std = float(da0.std(correction=0))

    # Train HQMM
    K_trained, hist = train_hqmm(
        p=p,
        K=K_torch,
        W=W,
        sym_idx=sym_idx,
        train_seqs=train_seqs,
        val_seqs=val_seqs,
        test_seqs=None,
        max_iter=max_iter,
        tol=tol,
        tau=tau,
        alpha=alpha,
        batch_size=batch_size,
        verbose=True,
        device=device
    )

    histories[label] = hist
    trained_models[label] = K_trained

    # Unique filenames for each match length
    prefix = f"{accession}_m{model_match_len}_tau_{tau:g}"

    model_path = model_dir / f"{prefix}.pt"
    history_path = results_dir / f"{prefix}_history.csv"

    # Save best-validation model immediately
    checkpoint = {
        "K": {
            key: value.detach().cpu().clone()
            for key, value in K_trained.items()
        },
        "p": p.detach().cpu().clone(),
        "W": W,
        "sym_idx": sym_idx,
        "name": str(name),
        "accession": str(accession),
        "original_match_len": match_len,
        "model_match_len": model_match_len,
        "tau": tau,
        "alpha": alpha,
        "batch_size": batch_size,
        "max_iter": max_iter,
        "tol": tol,
        "epochs_completed": len(hist) - 1
    }

    torch.save(checkpoint, model_path)
    hist.to_csv(history_path, index=False)

    print(f"Model saved: {model_path}", flush=True)

    # Evaluate trained HQMM
    K_ref = next(iter(K_trained.values()))
    p_eval = p.to(
        device=K_ref.device,
        dtype=K_ref.dtype
    )

    with torch.no_grad():
        da = description_accuracy_hqmm(
            p_eval, K_trained, W,
            sym_idx, test_seqs
        )

    trained_mean = float(da.mean())
    trained_std = float(da.std(correction=0))

    results[label] = {
        "match_len": model_match_len,
        "hidden_states": W,
        "tau": tau,
        "untrained_da_mean": untrained_mean,
        "untrained_da_std": untrained_std,
        "trained_da_mean": trained_mean,
        "trained_da_std": trained_std,
        "epochs_completed": len(hist) - 1
    }

    print(
        f"Untrained DA: {untrained_mean:.4f} "
        f"± {untrained_std:.4f}\n"
        f"Trained DA:   {trained_mean:.4f} "
        f"± {trained_std:.4f}",
        flush=True
    )

    # Save results after each configuration
    pd.DataFrame.from_dict(
        results, orient="index"
    ).rename_axis("setting").to_csv(
        results_dir / f"{accession}_match_len_comparison.csv"
    )

print("\nAll HQMM configurations completed.")


--- tau=0.1, n=n: tau=0.1, match_len=39
Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14656.90  train_da_mean 0.0023  val_da_mean 0.0033
  iter   1  batch   1/5  loss 219.7971  tau 0.100000
  iter   1  batch   2/5  loss 188.7776  tau 0.100000
  iter   1  batch   3/5  loss 212.6348  tau 0.100000
  iter   1  batch   4/5  loss 158.4182  tau 0.100000
  iter   1  batch   5/5  loss 178.2780  tau 0.100000
iter   1  train   176.4s  log-lik -11547.84  train_da_mean 0.2284  val_da_mean 0.2077
  iter   2  batch   1/5  loss 151.8979  tau 0.080000
  iter   2  batch   2/5  loss 152.2816  tau 0.080000
  iter   2  batch   3/5  loss 161.1720  tau 0.080000
  iter   2  batch   4/5  loss 140.7047  tau 0.080000
  iter   2  batch   5/5  loss 240.3141  tau 0.080000
iter   2  train   378.1s  log-lik -11152.37  train_da_mean 0.2541  val_da_mean 0.2284
  iter   3  batch   1/5  loss 158.9593  tau 0.064000
  iter   3  batch   2/5  loss 159.8178  tau 0.064000
  iter   3  batch   3/5 